# Stipple & Dilation Fills

Two approaches to filling shapes for pen plotter output: stipple dots (Poisson/grid/jittered sampling) and concentric inset dilation rings.

In [1]:
import penpal
from penpal.shading import stipple, dilation
import numpy as np

pw = penpal.pen_width(0.25)

## Stipple Methods Comparison
Poisson disk, jittered grid, regular grid, and random sampling inside a hexagon.

In [2]:
# Make a hexagon
def hexagon(cx, cy, r):
    angles = np.linspace(0, 2*np.pi, 7)
    return np.column_stack([cx + r*np.cos(angles), cy + r*np.sin(angles)])

d = penpal.Drawing(12, 6)
methods = ['poisson', 'jittered', 'grid', 'random']
colors = ['#264653', '#2a9d8f', '#e9c46a', '#e76f51']

for i, (method, color) in enumerate(zip(methods, colors)):
    cx = 1.5 + i * 3
    hex_pts = hexagon(cx, 3, 1.3)
    
    # Outline
    d.layer(f'hex_{i}', color='#aaa', linewidth=pw).add(penpal.Paths([hex_pts]))
    
    # Stipple fill
    p = stipple.stipple_polygon(hex_pts, density=0.15,
                                 dot_radius=0.03, method=method, seed=42)
    d.layer(f'stip_{i}', color=color, linewidth=pw).add(p)

d

## Stipple Rectangle
Quick fill of a rectangular region.

In [3]:
d = penpal.Drawing(8, 6)

p = stipple.stipple_rect(1, 1, 7, 5, density=0.2,
                          dot_radius=0.04, method='poisson', seed=42)
d.layer('dots', color='#1a1a2e', linewidth=pw).add(p)
d

## Dilation — Polygon Inset Rings
Concentric inset fills for any polygon shape.

In [4]:
d = penpal.Drawing(8, 8)

# Star polygon
angles = np.linspace(0, 2*np.pi, 11)
r_vals = [3, 1.5] * 5 + [3]
star = np.column_stack([np.array(r_vals)*np.cos(angles) + 4,
                        np.array(r_vals)*np.sin(angles) + 4])

p = dilation.dilate_polygon(star, n_rings=30, inward=True)
d.layer('star', color='#e63946', linewidth=pw).add(p)
d

## Dilation — Circle
Concentric circles shrinking inward.

In [5]:
d = penpal.Drawing(8, 8)

p = dilation.dilate_circle(center=(4, 4), radius=3.5,
                            n_rings=25, n_points=200)
d.layer('circles', color='#0f3460', linewidth=pw).add(p)
d

## Dilation — Rectangle
Concentric rounded-corner rectangles.

In [6]:
d = penpal.Drawing(10, 8)

p = dilation.dilate_rect(1, 1, 9, 7, n_rings=25, spacing=0.12)
d.layer('rect', color='#2a9d8f', linewidth=pw).add(p)
d

## Multi-Dilate — Two-Color Alternating
Split even/odd rings for a two-pen effect.

In [7]:
d = penpal.Drawing(8, 8)

# Irregular polygon
angles = np.linspace(0, 2*np.pi, 8)
rng = np.random.default_rng(42)
r = 2.5 + rng.uniform(-0.5, 0.5, 8)
blob = np.column_stack([r*np.cos(angles) + 4, r*np.sin(angles) + 4])
blob = np.vstack([blob, blob[0:1]])  # close it

groups = dilation.multi_dilate(blob, n_rings=20, alternating=True)
d.layer('even', color='#264653', linewidth=pw).add(groups[0])
d.layer('odd', color='#e9c46a', linewidth=pw).add(groups[1])
d